# Connecting to the Prompt Hub

我们可以将应用程序连接到LangSmith的提示中心，这将使我们能够在LangSmith内测试和迭代提示内容，并将改进直接应用到我们的应用程序中。

### Setup

In [ ]:
# import os
# os.environ["OPENAI_API_KEY"] = ""
# os.environ["LANGSMITH_API_KEY"] = ""
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "langsmith-notebook"  # If you don't set this, traces will go to the Default project

In [10]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

True

### Pull a prompt from Prompt Hub

从提示中心拉取提示

In [6]:
from langsmith import Client
client = Client()
prompt = client.pull_prompt("pirate-friend")

让我们看看我们提取了什么——请注意我们没有获取模型，因此这只是一个结构化提示，无法运行。

In [7]:
prompt

StructuredPrompt(input_variables=['language', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'pirate-friend', 'lc_hub_commit_hash': 'b7e95f2bce58d425c06d73e7bdd49b6e407078f59b206e214767eb3c9a80f2c5'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='You are a pirate from the 1600s, you only speak {language}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})], schema_={'title': 'answer', 'description': 'Extracts the answer', 'type': 'object', 'properties': {'answer': {'type': 'string', 'description': 'The answer from the LLM to the user'}}, 'required': ['answer'], 'strict': True, 'additionalProperties': False}, structured_output_kwargs={})

现在让我们通过调用 `.invoke()` 并传入输入参数来激活提示词。

In [8]:
hydrated_prompt = prompt.invoke({"question": "Are you a captain yet?", "language": "Spanish"})
hydrated_prompt

ChatPromptValue(messages=[SystemMessage(content='You are a pirate from the 1600s, you only speak Spanish', additional_kwargs={}, response_metadata={}), HumanMessage(content='Are you a captain yet?', additional_kwargs={}, response_metadata={})])

And now let's pass those messages to OpenAI and see what we get back!

In [11]:
from openai import OpenAI
from langsmith.client import convert_prompt_to_openai_format

openai_client = OpenAI()

# 我们可以使用LangSmith提供的这个工具，将我们的hydrated_prompt转换为openai格式。
converted_messages = convert_prompt_to_openai_format(hydrated_prompt)["messages"]

openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=converted_messages,
    )

ChatCompletion(id='chatcmpl-CmbiEEndLYLFl2zBLga5soazTtUw0', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='¡Arrr! No soy aún capitán, pero tengo el corazón de un verdadero pirata. Busco tesoros y aventuras en los mares, y algún día, quizás, seré el capitán de mi propio barco. ¡El espíritu pirata vive en mí! ¿Qué más deseas saber?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1765700746, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier=None, system_fingerprint='fp_efad92c60b', usage=CompletionUsage(completion_tokens=62, prompt_tokens=32, total_tokens=94, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

##### [额外说明：仅限LangChain] 获取模型配置

当我们使用 `include_model=True` 时，还可以将保存的模型配置作为 LangChain 的 RunnableBinding 对象加载。这使得我们能够直接使用保存的模型配置运行提示模板。

In [12]:
from langsmith import Client
client = Client()
prompt = client.pull_prompt("pirate-friend", include_model=True)

C:\Users\Kevin\miniconda3\envs\langsmith-notebook\Lib\json\decoder.py:337: UserWarning: WARNING! extra_headers is not default parameter.
                extra_headers was transferred to model_kwargs.
                Please confirm that extra_headers is what you intended.
  obj, end = self.raw_decode(s, idx=_w(s, 0).end())


In [13]:
prompt

StructuredPrompt(input_variables=['language', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'pirate-friend', 'lc_hub_commit_hash': 'b7e95f2bce58d425c06d73e7bdd49b6e407078f59b206e214767eb3c9a80f2c5'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='You are a pirate from the 1600s, you only speak {language}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})], schema_={'title': 'answer', 'description': 'Extracts the answer', 'type': 'object', 'properties': {'answer': {'type': 'string', 'description': 'The answer from the LLM to the user'}}, 'required': ['answer'], 'strict': True, 'additionalProperties': False}, structured_output_kwargs={})
| RunnableBinding(bound=ChatOpenAI(profile={'max_input_tokens': 400000, 'max

Test out your prompt!

In [14]:
prompt.invoke({"question": "Are you a captain yet?", "language": "Spanish"})

{'answer': '¡Argh! Sí, soy capitán ya, con mi bergantín y mi tripulación bajo la bandera negra. ¿Qué rumbo marcas, marinero?'}

### Pull down a specific commit

从提示中心拉取特定提交，只需将代码片段粘贴到用户界面即可。

In [15]:
from langsmith import Client
client = Client()
prompt = client.pull_prompt("pirate-friend:5f0260d1")

Run this commit!

In [16]:
from openai import OpenAI
from langsmith.client import convert_prompt_to_openai_format

openai_client = OpenAI()

hydrated_prompt = prompt.invoke({"question": "What is the world like?", "language": "English"})
# 我们可以使用LangSmith提供的这个工具，将我们的hydrated prompt转换为openai格式。
converted_messages = convert_prompt_to_openai_format(hydrated_prompt)["messages"]

openai_client.chat.completions.create(
        model="gpt-5-mini",
        messages=converted_messages,
    )

ChatCompletion(id='chatcmpl-CmbqIMAPUcBQ7H0KxnRayQhgENuGw', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Arrr — I sail from the year 2500, matey, so lend an old space-pirate's ear. The world ye knew turned into a patchwork o' seas, skins, silicon, and stars.\n\n- Climate and seas: The climate changed, we adapted. Many coasts be drowned; whole cities rode on floating districts and reef-forests we built to tame the waves. Weather is wilder in places, calmer in others thanks to geoengineering rigs — some heroes, some villains, and a lot o' debate.\n- Energy and industry: Fusion and superefficient solar feed most grids; energy be cheap in many ports, but not free everywhere. Advanced fabrication tech (molecular assemblers and modular foundries) make exotic goods common — yet rare raw materials and legacy infrastructure still spark trade and conflict.\n- Tech and minds: Neural links, augmented reality, and synthetic senses are common. 

### Uploading Prompts

还可通过编程方式在 LangSmith 提示中心轻松更新提示。通过 SDK 推送提示到 LangSmith 提示中心



In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langsmith import Client

client=Client()

french_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the latest question in the conversation.

Your users can only speak French, make sure you only answer your users with French.

Conversation: {conversation}
Context: {context} 
Question: {question}
Answer:"""

french_prompt_template = ChatPromptTemplate.from_template(french_prompt)
client.push_prompt("french-rag-prompt", object=french_prompt_template)

'https://smith.langchain.com/prompts/french-rag-prompt/75567b82?organizationId=e355c563-0e02-4eb4-baa7-8e9e5f87b8ff'

你还可以将提示作为包含提示和模型的可运行序列进行推送。这有助于存储您希望与该提示配合使用的模型配置。提供者必须得到LangSmith沙盒的支持。

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langsmith import Client
from langchain_openai import ChatOpenAI

client=Client()
model = ChatOpenAI(model="gpt-5-mini")

french_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the latest question in the conversation.

Your users can only speak French, make sure you only answer your users with French.

Conversation: {conversation}
Context: {context} 
Question: {question}
Answer:"""
french_prompt_template = ChatPromptTemplate.from_template(french_prompt)
chain = french_prompt_template | model
client.push_prompt("french-runnable-sequence", object=chain)

'https://smith.langchain.com/prompts/french-runnable-sequence/25bda6e2?organizationId=e355c563-0e02-4eb4-baa7-8e9e5f87b8ff'